# Notebook 06 — Event Study Drug-Approval-Level Dataset Build

**Purpose:** Construct a drug-approval-level analytical dataset for Stata event-study estimation of whether PDUFA (1992) changed the share of FDA-approved drugs classified as DEA-controlled substances.

**Row definition:** One row per unique approved original drug application (`ApplNo`), defined by:
1. Keep rows where `SubmissionStatus == 'AP'` (approved)
2. Keep rows where `SubmissionType == 'ORIG'` (original submissions only)
3. Deduplicate by `ApplNo`, keeping the earliest `SubmissionStatusDate`
4. Drop rows where `approval_year` is missing or 2026 (partial year)

**Pipeline position:** Sits downstream of notebooks 01–03 (backbone and DEA linkage) and notebook 05 (annual panel). Where notebook 05 produces a collapsed annual panel over the full submission-event frame, this notebook produces the underlying drug-level dataset with the full column set needed for Stata regressions and sub-sample analyses.

**Outputs (all to `data/event_study/`):**
- `event_study_drug_panel.csv` — drug-approval-level analytical dataset
- `event_study_drug_panel_DATA_DICTIONARY.md` — column-by-column data dictionary
- `event_study_summary_stats.csv` — annual summary table for quick diagnostics

**Key complication:** `data/intermediate/fda_dea_controlled_substance_linkage.csv` is a Git LFS pointer in this environment. The fallback is `data/intermediate/fda_dea_active_ingredient_linkage_audit.csv`, which is keyed by `ActiveIngredient_list` and contains all needed DEA columns. Detection and fallback are handled in Cell 4.

In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
from IPython.display import display


def find_repo_root(start: Path) -> Path:
    """Walk upward until the thesis repo root is found."""
    for candidate in [start, *start.parents]:
        if (candidate / "logs" / "thesis_context.md").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Could not locate econ580-thesis repo root.")


pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

ROOT = find_repo_root(Path.cwd().resolve())

# ── Input paths ──────────────────────────────────────────────────────────────
BACKBONE_PATH = ROOT / "data" / "processed"    / "fda_backbone.csv"
LINKED_PATH   = ROOT / "data" / "intermediate" / "fda_dea_controlled_substance_linkage.csv"
AUDIT_PATH    = ROOT / "data" / "intermediate" / "fda_dea_active_ingredient_linkage_audit.csv"

# ── Output paths ─────────────────────────────────────────────────────────────
EVENT_STUDY_DIR = ROOT / "data" / "event_study"
PANEL_OUT       = EVENT_STUDY_DIR / "event_study_drug_panel.csv"
DICT_OUT        = EVENT_STUDY_DIR / "event_study_drug_panel_DATA_DICTIONARY.md"
SUMMARY_OUT     = EVENT_STUDY_DIR / "event_study_summary_stats.csv"

# ── Policy constants ──────────────────────────────────────────────────────────
PDUFA_YEAR       = 1992
HATCHWAXMAN_YEAR = 1984

# Roman-numeral schedule ranking: I = most restrictive (1), V = least (5)
SCHEDULE_MAP = {"I": 1, "II": 2, "III": 3, "IV": 4, "V": 5}

# String ID columns that must never be cast to int
STRING_ID_COLS = ["ApplNo", "SubmissionNo"]

print(f"ROOT          : {ROOT}")
print(f"Output dir    : {EVENT_STUDY_DIR}")
print(f"Backbone path : {BACKBONE_PATH}")
print(f"Audit path    : {AUDIT_PATH}")

ROOT          : /Users/alexdelatorre/Desktop/econ580-thesis
Output dir    : /Users/alexdelatorre/Desktop/econ580-thesis/data/event_study
Backbone path : /Users/alexdelatorre/Desktop/econ580-thesis/data/processed/fda_backbone.csv
Audit path    : /Users/alexdelatorre/Desktop/econ580-thesis/data/intermediate/fda_dea_active_ingredient_linkage_audit.csv


## 1. Load and validate inputs

In [2]:
# ── Load backbone ─────────────────────────────────────────────────────────────
# ApplNo and SubmissionNo must remain strings (leading zeros are meaningful)
backbone_raw = pd.read_csv(
    BACKBONE_PATH,
    dtype={col: str for col in STRING_ID_COLS},
    low_memory=False,
)

# Strip whitespace from key categorical string columns
for col in ["SubmissionStatus", "SubmissionType", "ApplType_clean",
            "ReviewPriority_clean", "ActiveIngredient_list"]:
    if col in backbone_raw.columns:
        backbone_raw[col] = backbone_raw[col].astype("string").str.strip()

# Force string on ID columns (belt-and-suspenders)
for col in STRING_ID_COLS:
    backbone_raw[col] = backbone_raw[col].astype(str).str.strip()

print(f"Backbone shape : {backbone_raw.shape}")
print(f"Unique ApplNo  : {backbone_raw['ApplNo'].nunique():,}")
dup_keys = backbone_raw.duplicated(["ApplNo", "SubmissionType", "SubmissionNo"]).sum()
print(f"Duplicate submission-event keys : {dup_keys}  (expected 0)")

Backbone shape : (191265, 59)
Unique ApplNo  : 27,662
Duplicate submission-event keys : 0  (expected 0)


In [3]:
# ── DEA linkage: detect Git LFS pointer; fall back to ingredient audit ────────
# (Pattern reused from notebook 05)

def inspect_git_lfs_pointer(path: Path) -> dict:
    """Return metadata dict; is_git_lfs_pointer=True when the file is an LFS pointer."""
    with path.open("r", encoding="utf-8", errors="ignore") as handle:
        first_lines = [handle.readline().strip() for _ in range(3)]
    is_pointer = bool(first_lines) and first_lines[0].startswith(
        "version https://git-lfs.github.com/spec/v1"
    )
    return {
        "path": str(path),
        "exists": path.exists(),
        "is_git_lfs_pointer": is_pointer,
        "first_line": first_lines[0] if first_lines else "",
    }


linked_meta = inspect_git_lfs_pointer(LINKED_PATH)
print(f"Linkage file LFS pointer: {linked_meta['is_git_lfs_pointer']}")

if linked_meta["is_git_lfs_pointer"]:
    linkage_source = "ingredient_audit_fallback"
    print(
        "NOTE: row-level linkage CSV is a Git LFS pointer — using ingredient-level "
        "audit file as fallback.  All DEA match results are equivalent; the audit "
        "file is keyed by ActiveIngredient_list and covers the full backbone."
    )
    audit = pd.read_csv(AUDIT_PATH, low_memory=False)
else:
    linkage_source = "materialized_linkage_csv"
    print("NOTE: row-level linkage CSV is materialized. Loading directly.")
    audit = pd.read_csv(LINKED_PATH, low_memory=False)

# Coerce boolean flag columns (may read as object after CSV round-trip)
BOOL_COLS = [
    "dea_confident_controlled_match_flag",
    "dea_list_i_only_match_flag",
    "dea_possible_candidate_only_flag",
    "dea_uncertain_flag",
]
for col in BOOL_COLS:
    if col in audit.columns:
        audit[col] = audit[col].fillna(False).astype(bool)

print(f"\nAudit rows            : {len(audit):,}")
print(f"Audit key nulls        : {audit['ActiveIngredient_list'].isna().sum()}")
print(f"Audit key unique vals  : {audit['ActiveIngredient_list'].nunique():,}")
print(f"Linkage source mode    : {linkage_source}")

Linkage file LFS pointer: True
NOTE: row-level linkage CSV is a Git LFS pointer — using ingredient-level audit file as fallback.  All DEA match results are equivalent; the audit file is keyed by ActiveIngredient_list and covers the full backbone.

Audit rows            : 3,202
Audit key nulls        : 0
Audit key unique vals  : 3,202
Linkage source mode    : ingredient_audit_fallback


## 2. Filter, deduplicate, and date-clean

In [4]:
# ── Filtering with documented row counts ──────────────────────────────────────
n0 = len(backbone_raw)

# Step 1: approved submissions only
step1 = backbone_raw[backbone_raw["SubmissionStatus"] == "AP"].copy()
n1 = len(step1)

# Step 2: original submissions only
step2 = step1[step1["SubmissionType"] == "ORIG"].copy()
n2 = len(step2)

filter_log = pd.DataFrame([
    {"step": "0. Full backbone",                       "rows": n0, "dropped": 0},
    {"step": "1. Keep SubmissionStatus == 'AP'",        "rows": n1, "dropped": n0 - n1},
    {"step": "2. Keep SubmissionType == 'ORIG'",        "rows": n2, "dropped": n1 - n2},
])
display(filter_log)

,step,rows,dropped
0,0. Full backbone,191265,0
1,1. Keep SubmissionStatus == 'AP',190051,1214
2,2. Keep SubmissionType == 'ORIG',26268,163783


In [5]:
# ── Deduplication: keep earliest ORIG+AP row per ApplNo ──────────────────────
# Some applications have multiple ORIG+AP rows (e.g., a later re-approval under
# a different SubmissionNo). We keep the earliest approval date so the row
# represents the first market-entry event.

step2 = step2.copy()
step2["SubmissionStatusDate"] = pd.to_datetime(
    step2["SubmissionStatusDate"], errors="coerce"
)

n_before_dedup  = len(step2)
n_unique_applno = step2["ApplNo"].nunique()
n_dupes         = n_before_dedup - n_unique_applno

df = (
    step2
    .sort_values("SubmissionStatusDate", na_position="last")
    .drop_duplicates(subset="ApplNo", keep="first")
    .copy()
)

print(f"Before deduplication : {n_before_dedup:,} rows")
print(f"  Unique ApplNo      : {n_unique_applno:,}")
print(f"  Duplicate rows     : {n_dupes:,}  (retained earliest date per ApplNo)")
print(f"After deduplication  : {len(df):,} rows")

Before deduplication : 26,268 rows
  Unique ApplNo      : 26,076
  Duplicate rows     : 192  (retained earliest date per ApplNo)
After deduplication  : 26,076 rows


In [6]:
# ── Derive approval_year and approval_date; drop nulls and 2026 ───────────────

df["approval_date"] = df["SubmissionStatusDate"]
# Null out 1900-01-01 placeholder dates (persistent constraint)
df.loc[df["approval_date"] == pd.Timestamp("1900-01-01"), "approval_date"] = pd.NaT

df["approval_year"] = df["approval_date"].dt.year

n_null_year = df["approval_year"].isna().sum()
n_year_2026 = (df["approval_year"] == 2026).sum()
print(f"Null approval_year      : {n_null_year}")
print(f"2026 rows (partial year): {n_year_2026}")

df = df[df["approval_year"].notna() & (df["approval_year"] != 2026)].copy()
df["approval_year"] = df["approval_year"].astype(int)

n_final_premerge = len(df)
print(f"\nFinal rows (pre-DEA merge): {n_final_premerge:,}")
print(f"Year range               : {df['approval_year'].min()} – {df['approval_year'].max()}")

Null approval_year      : 2
2026 rows (partial year): 166

Final rows (pre-DEA merge): 25,908
Year range               : 1939 – 2025


## 3. DEA merge and confidence tier derivation

In [7]:
# ── Left join on ActiveIngredient_list ────────────────────────────────────────
# The audit table has one row per unique ActiveIngredient_list string,
# so validate="m:1" confirms no row inflation from the merge.

n_before_merge = len(df)
df = df.merge(audit, on="ActiveIngredient_list", how="left", validate="m:1")
assert len(df) == n_before_merge, (
    f"Row count changed after DEA merge: {n_before_merge} → {len(df)}"
)

# Re-coerce boolean flags post-merge (NaN arises for rows with no ingredient match)
for col in BOOL_COLS:
    if col in df.columns:
        df[col] = df[col].fillna(False).astype(bool)

print(f"Rows after DEA merge: {len(df):,}  (unchanged, as expected)")

Rows after DEA merge: 25,908  (unchanged, as expected)


In [8]:
# ── Confidence tier derivation ────────────────────────────────────────────────
# Assignment order matters: default first, most restrictive last.
# A row tagged confident_controlled also passes the list_i flag on some
# combination products; 'confident_scheduled' must win.

df["dea_confidence_tier"] = "no_dea_signal"

df.loc[df["ActiveIngredient_list"].isna(), "dea_confidence_tier"] = "no_ingredient_info"

df.loc[
    df["dea_possible_candidate_only_flag"]
    & ~df["dea_confident_controlled_match_flag"]
    & ~df["dea_list_i_only_match_flag"],
    "dea_confidence_tier",
] = "candidate_only"

df.loc[
    df["dea_list_i_only_match_flag"] & ~df["dea_confident_controlled_match_flag"],
    "dea_confidence_tier",
] = "list1"

df.loc[df["dea_confident_controlled_match_flag"], "dea_confidence_tier"] = "confident_scheduled"

print("DEA confidence tier distribution:")
display(
    df["dea_confidence_tier"]
    .value_counts()
    .rename_axis("tier")
    .reset_index(name="n")
)

DEA confidence tier distribution:


,tier,n
0,no_dea_signal,22366
1,confident_scheduled,2067
2,no_ingredient_info,1155
3,candidate_only,201
4,list1,119


## 4. Construct derived columns

In [9]:
# ── FDA identification and application-type dummies ───────────────────────────

df["ApplType"] = df["ApplType_clean"]
df["is_nda"]   = (df["ApplType_clean"] == "NDA").astype(int)
df["is_anda"]  = (df["ApplType_clean"] == "ANDA").astype(int)
df["is_bla"]   = (df["ApplType_clean"] == "BLA").astype(int)

print("ApplType distribution:")
display(
    df["ApplType"]
    .value_counts(dropna=False)
    .rename_axis("ApplType")
    .reset_index(name="n")
)

ApplType distribution:


,ApplType,n
0,ANDA,19022
1,NDA,5277
2,UNKNOWN,1149
3,BLA,460


In [10]:
# ── DEA controlled-substance dummies ─────────────────────────────────────────

# Primary: conservative confident-scheduled tier only
df["is_controlled_substance"] = (
    df["dea_confidence_tier"] == "confident_scheduled"
).astype(int)

# Broadest: any DEA signal including uncertain candidates
df["is_controlled_substance_broad"] = (
    df["dea_confidence_tier"].isin(["confident_scheduled", "list1", "candidate_only"])
).astype(int)

# Intermediate: confident + List I chemicals
df["is_controlled_or_list1"] = (
    df["dea_confidence_tier"].isin(["confident_scheduled", "list1"])
).astype(int)

# Pass-through DEA match metadata
df["dea_substance_name_list"] = df["dea_confident_substance_names"]
df["dea_schedule"]            = df["dea_confident_current_schedules"]

print(f"is_controlled_substance       : {df['is_controlled_substance'].sum():,}")
print(f"is_controlled_or_list1        : {df['is_controlled_or_list1'].sum():,}")
print(f"is_controlled_substance_broad : {df['is_controlled_substance_broad'].sum():,}")

is_controlled_substance       : 2,067
is_controlled_or_list1        : 2,186
is_controlled_substance_broad : 2,387


In [11]:
# ── Schedule parsing ──────────────────────────────────────────────────────────
# dea_confident_current_schedules uses ' | ' (pipe with spaces) as separator
# for multi-substance matches, e.g. 'II | IV'.
# Some DEA references prefix schedules with 'C' (e.g., 'CIII') — strip it.

def _parse_schedule_tokens(s: str) -> list:
    """Split a schedule string on '|' and return a list of normalised Roman-numeral tokens."""
    if not isinstance(s, str) or not s.strip():
        return []
    return [re.sub(r"^C", "", t.strip()) for t in s.split("|")]


def parse_schedule_highest(s) -> object:
    """Return the most restrictive (lowest-numbered) schedule from a pipe-separated string."""
    if pd.isna(s):
        return None
    tokens = _parse_schedule_tokens(str(s))
    valid  = [t for t in tokens if t in SCHEDULE_MAP]
    if not valid:
        return None
    return min(valid, key=lambda x: SCHEDULE_MAP[x])


def schedule_contains(s, target: str) -> int:
    """Return 1 if *target* schedule appears in pipe-separated *s*, else 0."""
    if pd.isna(s):
        return 0
    return int(target in _parse_schedule_tokens(str(s)))


df["dea_schedule_highest"] = df["dea_schedule"].apply(parse_schedule_highest)

for sched in ["I", "II", "III", "IV", "V"]:
    df[f"is_schedule_{sched}"] = df["dea_schedule"].apply(
        lambda s, sr=sched: schedule_contains(s, sr)
    )

print("dea_schedule_highest distribution:")
display(
    df["dea_schedule_highest"]
    .value_counts(dropna=False)
    .rename_axis("schedule")
    .reset_index(name="n")
)

dea_schedule_highest distribution:


,schedule,n
0,NaN,23841
1,II,994
2,IV,769
3,III,189
4,V,108
5,I,7


In [12]:
# ── Multi-ingredient and combination-product flags ────────────────────────────
# ActiveIngredient_list uses semicolons to delimit multiple ingredients.

def count_ingredient_tokens(s) -> int:
    if pd.isna(s) or not str(s).strip():
        return 0
    return len([t for t in str(s).split(";") if t.strip()])


df["is_multi_ingredient"] = df["ActiveIngredient_list"].apply(
    lambda s: int(count_ingredient_tokens(s) >= 2)
)

df["has_controlled_ingredient_in_combo"] = (
    (df["is_controlled_substance"] == 1) & (df["is_multi_ingredient"] == 1)
).astype(int)

print(f"is_multi_ingredient                   : {df['is_multi_ingredient'].sum():,}")
print(f"has_controlled_ingredient_in_combo    : {df['has_controlled_ingredient_in_combo'].sum():,}")

is_multi_ingredient                   : 3,057
has_controlled_ingredient_in_combo    : 592


In [13]:
# ── Policy timing variables ───────────────────────────────────────────────────

def pdufa_era_label(year) -> object:
    if pd.isna(year):
        return None
    y = int(year)
    if y < 1984:  return "pre_hatchwaxman"
    if y < 1993:  return "post_hatchwaxman_pre_pdufa"
    if y <= 1997: return "pdufa_I"
    if y <= 2002: return "pdufa_II"
    if y <= 2007: return "pdufa_III"
    if y <= 2012: return "pdufa_IV"
    if y <= 2017: return "pdufa_V"
    if y <= 2022: return "pdufa_VI"
    return "pdufa_VII"


df["post_pdufa"]       = (df["approval_year"] >= 1993).astype(int)
df["post_hatchwaxman"] = (df["approval_year"] >= 1984).astype(int)
df["pdufa_era"]        = df["approval_year"].apply(pdufa_era_label)
df["event_time"]       = df["approval_year"] - PDUFA_YEAR

print("pdufa_era distribution:")
display(
    df["pdufa_era"]
    .value_counts()
    .rename_axis("era")
    .reset_index(name="n")
)

pdufa_era distribution:


,era,n
0,pdufa_VI,4380
1,pre_hatchwaxman,4270
2,pdufa_V,3464
3,post_hatchwaxman_pre_pdufa,3031
4,pdufa_IV,2761
5,pdufa_VII,2534
6,pdufa_III,2281
7,pdufa_II,1662
8,pdufa_I,1525


## 5. Diagnostics and integrity checks

In [14]:
print("=" * 60)
print("FINAL DATASET PROFILE")
print("=" * 60)
print(f"Rows                  : {len(df):,}")
print(f"Unique ApplNo         : {df['ApplNo'].nunique():,}")
print(f"Year range            : {df['approval_year'].min()} – {df['approval_year'].max()}")
print()

print("ApplType breakdown:")
display(df["ApplType"].value_counts(dropna=False).to_frame("n"))

print("\nDEA tier breakdown:")
display(df["dea_confidence_tier"].value_counts().to_frame("n"))

print("\nSchedule breakdown (highest restrictive schedule):")
display(df["dea_schedule_highest"].value_counts(dropna=False).to_frame("n"))

print("\nis_controlled_substance (confident_scheduled only) :",
      df["is_controlled_substance"].sum())
print("is_controlled_or_list1                            :",
      df["is_controlled_or_list1"].sum())
print("is_controlled_substance_broad                     :",
      df["is_controlled_substance_broad"].sum())

print("\n--- Integrity assertions ---")
assert (df["approval_year"] == 2026).sum() == 0, "FAIL: 2026 rows present"
print("PASS: no 2026 rows")

# Check string dtype — pandas may use StringDtype ('str') or object depending on version
assert not pd.api.types.is_numeric_dtype(df["ApplNo"]), \
    f"FAIL: ApplNo is numeric dtype {df['ApplNo'].dtype}"
print(f"PASS: ApplNo is non-numeric string dtype ({df['ApplNo'].dtype})")

assert len(df) == df["ApplNo"].nunique(), (
    f"FAIL: {len(df)} rows but {df['ApplNo'].nunique()} unique ApplNo"
)
print("PASS: ApplNo is unique (no post-dedup duplicates)")

assert df["is_controlled_substance"].sum() > 0, "FAIL: zero controlled substances found"
print("PASS: at least one controlled substance row found")

FINAL DATASET PROFILE
Rows                  : 25,908
Unique ApplNo         : 25,908
Year range            : 1939 – 2025

ApplType breakdown:


,n
ApplType,
ANDA,19022
NDA,5277
UNKNOWN,1149
BLA,460



DEA tier breakdown:


,n
dea_confidence_tier,
no_dea_signal,22366
confident_scheduled,2067
no_ingredient_info,1155
candidate_only,201
list1,119



Schedule breakdown (highest restrictive schedule):


,n
dea_schedule_highest,
NaN,23841
II,994
IV,769
III,189
V,108
I,7



is_controlled_substance (confident_scheduled only) : 2067
is_controlled_or_list1                            : 2186
is_controlled_substance_broad                     : 2387

--- Integrity assertions ---
PASS: no 2026 rows
PASS: ApplNo is non-numeric string dtype (str)
PASS: ApplNo is unique (no post-dedup duplicates)
PASS: at least one controlled substance row found


## 6. Column selection, summary stats, and export

In [15]:
# ── Select and order final columns ────────────────────────────────────────────

FINAL_COLS = [
    # FDA identification and timing
    "ApplNo",
    "ApplType",
    "approval_year",
    "approval_date",
    "SponsorName",
    "DrugName_list",
    "ActiveIngredient_list",
    # Regulatory pathway
    "ReviewPriority_raw",
    "ReviewPriority_clean",
    "SubmissionClassCode",
    "is_nda",
    "is_anda",
    "is_bla",
    # DEA controlled substance
    "is_controlled_substance",
    "is_controlled_substance_broad",
    "is_controlled_or_list1",
    "dea_schedule",
    "dea_schedule_highest",
    "is_schedule_I",
    "is_schedule_II",
    "is_schedule_III",
    "is_schedule_IV",
    "is_schedule_V",
    "is_multi_ingredient",
    "has_controlled_ingredient_in_combo",
    "dea_confidence_tier",
    "dea_substance_name_list",
    # Policy timing
    "post_pdufa",
    "post_hatchwaxman",
    "pdufa_era",
    "event_time",
]

# Verify all columns exist before slicing
missing_cols = [c for c in FINAL_COLS if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing columns in df: {missing_cols}")

panel = df[FINAL_COLS].copy().sort_values("approval_year").reset_index(drop=True)

print(f"Final panel shape: {panel.shape}")
display(panel.head(3))

Final panel shape: (25908, 31)


,ApplNo,ApplType,approval_year,approval_date,SponsorName,DrugName_list,ActiveIngredient_list,ReviewPriority_raw,ReviewPriority_clean,SubmissionClassCode,is_nda,is_anda,is_bla,is_controlled_substance,is_controlled_substance_broad,is_controlled_or_list1,dea_schedule,dea_schedule_highest,is_schedule_I,is_schedule_II,is_schedule_III,is_schedule_IV,is_schedule_V,is_multi_ingredient,has_controlled_ingredient_in_combo,dea_confidence_tier,dea_substance_name_list,post_pdufa,post_hatchwaxman,pdufa_era,event_time
0,000552,NDA,1939,1939-02-09,ASPEN GLOBAL INC,LIQUAEMIN SODIUM; LIQUAEMIN LOCK FLUSH; HEPARI...,HEPARIN SODIUM,UNKNOWN,UNKNOWN,UNKNOWN,1,0,0,0,0,0,NaN,NaN,0,0,0,0,0,0,0,no_dea_signal,NaN,0,0,pre_hatchwaxman,-53
1,001645,UNKNOWN,1939,1939-10-06,NaN,NaN,<NA>,NaN,UNKNOWN,NaN,0,0,0,0,0,0,NaN,NaN,0,0,0,0,0,0,0,no_ingredient_info,NaN,0,0,pre_hatchwaxman,-53
2,001546,NDA,1939,1939-10-02,MERCK SHARP DOHME,GUANIDINE HYDROCHLORIDE,GUANIDINE HYDROCHLORIDE,UNKNOWN,UNKNOWN,UNKNOWN,1,0,0,0,0,0,NaN,NaN,0,0,0,0,0,0,0,no_dea_signal,NaN,0,0,pre_hatchwaxman,-53


In [16]:
# ── Annual summary stats table ────────────────────────────────────────────────

summary_stats = (
    panel
    .groupby("approval_year")
    .agg(
        n_approvals             = ("ApplNo",                        "count"),
        n_controlled            = ("is_controlled_substance",       "sum"),
        n_controlled_broad      = ("is_controlled_substance_broad", "sum"),
        n_nda                   = ("is_nda",                        "sum"),
        n_anda                  = ("is_anda",                       "sum"),
        n_schedule_I            = ("is_schedule_I",                 "sum"),
        n_schedule_II           = ("is_schedule_II",                "sum"),
        n_schedule_III          = ("is_schedule_III",               "sum"),
        n_schedule_IV           = ("is_schedule_IV",                "sum"),
        n_schedule_V            = ("is_schedule_V",                 "sum"),
    )
    .reset_index()
)

summary_stats["share_controlled"]       = (
    summary_stats["n_controlled"] / summary_stats["n_approvals"]
)
summary_stats["share_controlled_broad"] = (
    summary_stats["n_controlled_broad"] / summary_stats["n_approvals"]
)

# Reorder columns
summary_stats = summary_stats[[
    "approval_year", "n_approvals",
    "n_controlled", "share_controlled",
    "n_controlled_broad", "share_controlled_broad",
    "n_nda", "n_anda",
    "n_schedule_I", "n_schedule_II", "n_schedule_III", "n_schedule_IV", "n_schedule_V",
]]

print(f"Summary stats shape: {summary_stats.shape}  (one row per year)")
print(f"\nTotal approvals across all years: {summary_stats['n_approvals'].sum():,}  (should match panel rows)")
assert summary_stats["n_approvals"].sum() == len(panel), "Row count mismatch!"
print("PASS: annual totals sum to panel row count")

print("\nSample (1985–2000):")
display(
    summary_stats[
        (summary_stats["approval_year"] >= 1985) &
        (summary_stats["approval_year"] <= 2000)
    ].to_string(index=False)
)

Summary stats shape: (87, 13)  (one row per year)

Total approvals across all years: 25,908  (should match panel rows)
PASS: annual totals sum to panel row count

Sample (1985–2000):


' approval_year  n_approvals  n_controlled  share_controlled  n_controlled_broad  share_controlled_broad  n_nda  n_anda  n_schedule_I  n_schedule_II  n_schedule_III  n_schedule_IV  n_schedule_V\n          1985          393            42            0.1069                  50                  0.1272     82     291             0             16               0             26             0\n          1986          478            54            0.1130                  58                  0.1213     78     363             0             20               1             33             0\n          1987          501            59            0.1178                  62                  0.1238     58     390             0             10               0             49             0\n          1988          455            49            0.1077                  51                  0.1121     59     359             0             18               0             31             0\n          1989          250  

In [17]:
# ── Create output directory and export all files ──────────────────────────────

EVENT_STUDY_DIR.mkdir(parents=True, exist_ok=True)

# 1. Drug-approval-level panel
panel.to_csv(PANEL_OUT, index=False)
print(f"Exported panel          : {PANEL_OUT}")
print(f"  rows x cols           : {panel.shape}")

# 2. Annual summary stats
summary_stats.to_csv(SUMMARY_OUT, index=False)
print(f"Exported summary stats  : {SUMMARY_OUT}")
print(f"  rows x cols           : {summary_stats.shape}")

# 3. Data dictionary (markdown)
COLUMN_DEFS = [
    ("ApplNo",                          "str",   "FDA application number. String identifier — leading zeros are preserved. One unique value per row."),
    ("ApplType",                         "str",   "Application type (NDA, ANDA, BLA, NDA authorized generic, etc.). Derived from ApplType_clean in the backbone."),
    ("approval_year",                    "int",   "Calendar year of the first original approved submission for this application. Extracted from SubmissionStatusDate."),
    ("approval_date",                    "date",  "Full date of the first original approved submission. Null if date was missing or placeholder 1900-01-01."),
    ("SponsorName",                      "str",   "FDA applicant/sponsor name as recorded in the Drugs@FDA Applications table."),
    ("DrugName_list",                    "str",   "Drug/product name(s) associated with this application, semicolon-delimited if multiple products share the same application."),
    ("ActiveIngredient_list",            "str",   "Active ingredient(s) associated with this application, semicolon-delimited. Join key for DEA linkage."),
    ("ReviewPriority_raw",               "str",   "Review priority as recorded in Submissions table (raw, may include 'UNKNOWN' or blanks)."),
    ("ReviewPriority_clean",             "str",   "Review priority after backbone-level cleaning (S=standard, P=priority, UNKNOWN). Quality varies over time."),
    ("SubmissionClassCode",              "str",   "Submission class code indicating the type of new drug application (e.g., Type 1 New Molecular Entity)."),
    ("is_nda",                           "int",   "Dummy: 1 if ApplType == 'NDA', 0 otherwise."),
    ("is_anda",                          "int",   "Dummy: 1 if ApplType == 'ANDA' (generic), 0 otherwise."),
    ("is_bla",                           "int",   "Dummy: 1 if ApplType == 'BLA' (biologic), 0 otherwise."),
    ("is_controlled_substance",          "int",   "PRIMARY DEA dummy. 1 if dea_confidence_tier == 'confident_scheduled' (exact or root-normalized match to a CSA-scheduled ingredient), 0 otherwise. Conservative definition."),
    ("is_controlled_substance_broad",    "int",   "BROAD DEA dummy. 1 if tier is confident_scheduled, list1, or candidate_only. Widest net; use as sensitivity check."),
    ("is_controlled_or_list1",           "int",   "INTERMEDIATE DEA dummy. 1 if tier is confident_scheduled or list1 (DEA List I chemical). Note: List I chemicals are precursors, not necessarily CSA scheduled substances."),
    ("dea_schedule",                     "str",   "DEA schedule(s) for confidently matched ingredients, pipe-delimited (e.g., 'II' or 'II | IV'). Null if no confident match."),
    ("dea_schedule_highest",             "str",   "Most restrictive (lowest Roman numeral) schedule among matched ingredients. E.g., if an app has both Schedule II and IV ingredients, this is 'II'. Null if no confident match."),
    ("is_schedule_I",                    "int",   "Dummy: 1 if any confidently matched ingredient is Schedule I."),
    ("is_schedule_II",                   "int",   "Dummy: 1 if any confidently matched ingredient is Schedule II."),
    ("is_schedule_III",                  "int",   "Dummy: 1 if any confidently matched ingredient is Schedule III."),
    ("is_schedule_IV",                   "int",   "Dummy: 1 if any confidently matched ingredient is Schedule IV."),
    ("is_schedule_V",                    "int",   "Dummy: 1 if any confidently matched ingredient is Schedule V."),
    ("is_multi_ingredient",              "int",   "Dummy: 1 if ActiveIngredient_list contains 2 or more semicolon-delimited tokens (combination product)."),
    ("has_controlled_ingredient_in_combo","int",  "Dummy: 1 if is_controlled_substance == 1 AND is_multi_ingredient == 1 (combination product with at least one controlled ingredient)."),
    ("dea_confidence_tier",              "str",   "DEA linkage confidence tier. Values: confident_scheduled, list1, candidate_only, no_ingredient_info, no_dea_signal. Carried for audit."),
    ("dea_substance_name_list",          "str",   "DEA substance name(s) for confidently matched ingredients. Null if no confident match."),
    ("post_pdufa",                       "int",   "Policy dummy: 1 if approval_year >= 1993 (PDUFA enacted Oct 1992; 1993 is first full implementation year)."),
    ("post_hatchwaxman",                 "int",   "Policy dummy: 1 if approval_year >= 1984 (Hatch-Waxman Act)."),
    ("pdufa_era",                        "str",   "Categorical policy era. Values: pre_hatchwaxman (<1984), post_hatchwaxman_pre_pdufa (1984-1992), pdufa_I (1993-1997), pdufa_II (1998-2002), pdufa_III (2003-2007), pdufa_IV (2008-2012), pdufa_V (2013-2017), pdufa_VI (2018-2022), pdufa_VII (2023+)."),
    ("event_time",                       "int",   "Event-study running variable: approval_year - 1992. Negative values are pre-PDUFA, zero is 1992, positive values are post-PDUFA."),
]

dict_lines = [
    "# Data Dictionary: event_study_drug_panel.csv\n",
    "\n",
    f"**Generated by:** `code/notebooks/06_event_study_dataset_build.ipynb`  \n",
    "**Source inputs:** `data/processed/fda_backbone.csv` + `data/intermediate/fda_dea_active_ingredient_linkage_audit.csv`  \n",
    "\n",
    "**Row definition:** One row per unique FDA application (`ApplNo`), defined as the earliest original approved "
    "submission event (`SubmissionType == ORIG`, `SubmissionStatus == AP`). Applications with multiple ORIG+AP "
    "rows are deduplicated by keeping the earliest `SubmissionStatusDate`.  \n",
    "\n",
    f"**Total rows:** {len(panel):,}  \n",
    f"**Year range:** {panel['approval_year'].min()} – {panel['approval_year'].max()}  \n",
    "\n",
    "**Important caveats:**\n",
    "- DEA linkage is current (2026) and ingredient-level, not historical product-level scheduling.  \n",
    "- `dea_confidence_tier == 'no_dea_signal'` rows are coded 0 for all controlled-substance dummies — they are NOT dropped.  \n",
    "- `List I` chemicals (precursors) are not CSA-scheduled substances — keep `is_controlled_or_list1` as a sensitivity, not a main outcome.  \n",
    "- 2026 is excluded (partial calendar year from the March 2026 FDA extract).  \n",
    "\n",
    "---\n",
    "\n",
    "| Column | Type | Description |\n",
    "|--------|------|-------------|\n",
]
for col, dtype, desc in COLUMN_DEFS:
    dict_lines.append(f"| `{col}` | {dtype} | {desc} |\n")

with open(DICT_OUT, "w") as f:
    f.writelines(dict_lines)

print(f"Exported data dictionary: {DICT_OUT}")

Exported panel          : /Users/alexdelatorre/Desktop/econ580-thesis/data/event_study/event_study_drug_panel.csv
  rows x cols           : (25908, 31)


Exported summary stats  : /Users/alexdelatorre/Desktop/econ580-thesis/data/event_study/event_study_summary_stats.csv
  rows x cols           : (87, 13)
Exported data dictionary: /Users/alexdelatorre/Desktop/econ580-thesis/data/event_study/event_study_drug_panel_DATA_DICTIONARY.md


## Summary

### What was built

This notebook constructed a drug-approval-level analytical dataset for event-study estimation of PDUFA's effect on the composition of FDA-approved drugs.

**Filter funnel:**
- Started from the full FDA backbone (~191,000 submission events)
- Filtered to `SubmissionStatus == 'AP'` and `SubmissionType == 'ORIG'`
- Deduplicated by `ApplNo` (kept earliest approval date)
- Dropped rows with missing year or year == 2026

**DEA linkage:** The row-level linkage CSV was a Git LFS pointer; the ingredient-level audit file was used as fallback. Confidence tiers were derived in precedence order: `confident_scheduled` > `list1` > `candidate_only` > `no_ingredient_info` > `no_dea_signal`.

**Three output files** written to `data/event_study/`:
- `event_study_drug_panel.csv` — main dataset for Stata regression
- `event_study_drug_panel_DATA_DICTIONARY.md` — column definitions
- `event_study_summary_stats.csv` — annual diagnostic table

### Key limitations

- DEA schedules are **current** (2026), not historical schedule-at-approval. A drug approved in 1980 may have been scheduled differently at that time.
- Linkage is **ingredient-level**, not product-specific. Preparation-specific scheduling distinctions (e.g., codeine in low-dose combinations) are not resolved.
- The dataset covers **observed approvals** only — failed or withdrawn applications are not in Drugs@FDA.

### Next steps

1. Load `event_study_drug_panel.csv` into Stata for event-study regression estimation
2. Primary specification: outcome = `is_controlled_substance`, running variable = `event_time`, break year = 1992
3. Sensitivities: `is_controlled_or_list1`, `is_controlled_substance_broad`; NDA-only subsample (`is_nda == 1`)
4. Consider building a therapeutic-class-by-year panel from this file for heterogeneity analysis